# Notebook 02 — Model Training
Fine-tunes DistilBERT for binary prompt injection classification.

**Architecture:** distilbert-base-uncased + classification head
**Loss:** Class-weighted CrossEntropy (to boost injection recall)
**Optimizer:** AdamW with cosine LR schedule + warmup

In [ ]:
import os
import sys
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

BASE = os.getcwd()
DATA_DIR = os.path.join(BASE, "data")
MODEL_DIR = os.path.join(BASE, "models", "distilbert-injection")

os.makedirs(DATA_DIR, exist_ok=True)
print("Workspace:", BASE)

## 1. Preprocess & Tokenise

In [ ]:
train_path = os.path.join(DATA_DIR, "train.csv")
val_path = os.path.join(DATA_DIR, "val.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

for p in [train_path, val_path, test_path]:
    print(f"{os.path.basename(p)} exists: {os.path.exists(p)}")

if os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print("\nTrain shape:", train_df.shape)
    print("Test shape:", test_df.shape)
    display(train_df.head())
else:
    print("train.csv not found yet. It will be created by train.py.")

## 2. Compute Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

if os.path.exists(train_path):
    labels = train_df["label"].values
    classes = np.unique(labels)
    weights = compute_class_weight("balanced", classes=classes, y=labels)
    class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
    print("Class weights:", class_weights)
else:
    print("Run the training cell first to generate splits, then re-run this cell.")

## 3. Train

In [ ]:
print("Running full training pipeline via train.py ...")
result = subprocess.run(
    [sys.executable, "train.py"],
    capture_output=True,
    text=True,
    cwd=BASE
)
print("Return code:", result.returncode)
print("\n----- STDOUT (last 120 lines) -----")
print("\n".join(result.stdout.splitlines()[-120:]))
if result.stderr.strip():
    print("\n----- STDERR (last 80 lines) -----")
    print("\n".join(result.stderr.splitlines()[-80:]))

if result.returncode != 0:
    raise RuntimeError("train.py failed. See stderr above.")

## 4. Training Curves

In [ ]:
eda_path = os.path.join(DATA_DIR, "eda_overview.png")
cm_path = os.path.join(DATA_DIR, "confusion_matrix.png")

print("Artifacts after training:")
for p in [eda_path, cm_path, MODEL_DIR]:
    print(f"- {p}: {os.path.exists(p)}")

if os.path.exists(eda_path):
    plt.figure(figsize=(12, 4))
    img = plt.imread(eda_path)
    plt.imshow(img)
    plt.axis("off")
    plt.title("EDA Overview")
    plt.show()

if os.path.exists(cm_path):
    plt.figure(figsize=(6, 5))
    img = plt.imread(cm_path)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix")
    plt.show()

print("Training notebook complete. For richer evaluation outputs, run 03_evaluation.ipynb.")